In [1]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Copy the exact list of groups from your tuning notebook here
# (We need this list so the final exam loop knows which folders to open)
with open("C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt_Beta/data/processed/go_group_assignment_v3.json", "r") as f:
    _blueprint_data = json.load(f)
TUNE_GROUPS = list(_blueprint_data["groups"].keys())

class ProteinDataset(Dataset):
    def __init__(self, indices, feature_matrix, label_matrix):
        self.indices        = indices
        self.feature_matrix = feature_matrix
        self.label_matrix   = label_matrix
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        i = self.indices[idx]
        features = torch.tensor(self.feature_matrix[i], dtype=torch.float32)
        label    = torch.tensor(self.label_matrix[i],   dtype=torch.float32)
        return features, label

class ProteinMLP(nn.Module):
    def __init__(self, input_dim=498, num_classes=128, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.network(x)

def load_group_data(group_name, processed_dir):
    safe_name = group_name.replace(":", "_").replace("/", "_")
    group_dir = os.path.join(processed_dir, safe_name)
    with open(os.path.join(group_dir, "accessions.json")) as f:
        accessions = json.load(f)
    with np.load(os.path.join(group_dir, "labels.npz")) as archive:
        label_matrix = archive["labels"]
    feature_matrix = np.load(os.path.join(group_dir, "features.npy"))
    with open(os.path.join(group_dir, "terms.json")) as f:
        go_term_list = json.load(f)
    return accessions, feature_matrix, label_matrix, go_term_list

print("✓ New evaluation notebook brain successfully initialized!")

✓ New evaluation notebook brain successfully initialized!


In [2]:
"""
================================================================================
NeuralProt — Final Exam Evaluator (Test Set Scorer)
================================================================================
What this code does:
1. It opens your saved threshold file to see the strictness rules we found.
2. It loops through your groups and loads the hidden 'test_idx.json' checklists.
3. It passes those unseen test numbers through your model brain layers.
4. It applies your saved custom thresholds unchanged to calculate your true score.
"""

# ── 1. CONFIGURATION PATHS ────────────────────────────────────────────────────
PROCESSED_DIR = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt_Beta/data/processed/processed_data"
MODEL_DIR = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt_Beta/backend/models"
THRESHOLD_MAP_PATH = os.path.join(MODEL_DIR, "threshold_results.json")

print("=" * 90)
print("📝 Launching Final Exam Grader Across All Folders")
print("=" * 90)

# Verify that our threshold rules exist
if not os.path.exists(THRESHOLD_MAP_PATH):
    print("❌ ERROR: You must run your threshold tuning notebook blocks first!")
else:
    with open(THRESHOLD_MAP_PATH, "r") as f:
        saved_thresholds_dict = json.load(f)

    final_test_report = []

    # ── 2. THE GRADER ENGINE LOOP ──
    for group_name in TUNE_GROUPS:
        # Clean out colons and slashes so our paths match your flat folders
        safe_name = group_name.replace(":", "_").replace("/", "_")
        
        group_dir = os.path.join(PROCESSED_DIR, safe_name)
        model_path = os.path.join(MODEL_DIR, f"{safe_name}_best.pt")
        test_idx_path = os.path.join(group_dir, "test_idx.json")
        
        # Skip if files are missing
        if not os.path.exists(model_path) or not os.path.exists(test_idx_path):
            continue
            
        try:
            # Load your data files
            with open(test_idx_path, "r") as f:
                test_indices = json.load(f)
                
            with np.load(os.path.join(group_dir, "labels.npz")) as archive:
                label_matrix = archive["labels"]
            feature_matrix = np.load(os.path.join(group_dir, "features.npy"))
            with open(os.path.join(group_dir, "terms.json")) as f:
                go_term_list = json.load(f)
                
            # Pull the pre-saved custom threshold percentage we found during the quiz
            group_threshold_settings = saved_thresholds_dict.get(group_name, {})
            custom_sweet_spot = group_threshold_settings.get("optimal_threshold", {}).get("threshold", 0.50)
            
            # Wrap the test indices inside our streaming dataset box
            test_dataset = ProteinDataset(test_indices, feature_matrix, label_matrix)
            test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)
            
            # Load the saved model brain weights
            checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=False)
            model = ProteinMLP(num_classes=len(go_term_list)).to(DEVICE)
            model.load_state_dict(checkpoint["model_state"])
            model.eval()
            
            # Collect probabilities from the final exam pile
            all_probs = []
            all_targets = []
            
            with torch.no_grad():
                for features, labels in test_loader:
                    features = features.to(DEVICE)
                    logits = model(features)
                    probs = torch.sigmoid(logits)
                    all_probs.append(probs.cpu().numpy())
                    all_targets.append(labels.numpy())
                    
            test_probs = np.vstack(all_probs)
            test_targets = np.vstack(all_targets)
            
            # Apply your saved custom threshold percentage unchanged!
            test_preds = (test_probs >= custom_sweet_spot).astype(np.float32)
            
            # Calculate the final true exam scores
            final_f1 = f1_score(test_targets, test_preds, average="macro", zero_division=0)
            
            final_test_report.append({
                "name": safe_name,
                "f1": final_f1,
                "proteins": len(test_indices),
                "threshold": custom_sweet_spot
            })
            
        except Exception as e:
            print(f" ❌ Failed to grade final exam for {safe_name}: {e}")

    # ── 3. PRINT THE FINAL LEDGER SORTED BY SCORE ──
    print(f"  {'Model Group Folder Name':<42} | {'Exam Samples':<12} | {'Used Threshold':<14} | {'True Final F1':<8}")
    print("-" * 90)
    
    # Sort from highest score to lowest score so you can see your true victories
    for r in sorted(final_test_report, key=lambda x: x["f1"], reverse=True):
        short_name = r["name"] if len(r["name"]) <= 42 else r["name"][:39] + "..."
        print(f"  {short_name:<42} | {r['proteins']:<12} | {r['threshold']:<14.2f} | {r['f1']:<8.4f}")
        
    print("=" * 90)
    print("✓ Final exam scoring complete. These are your true, publication-ready real world scores!")
    print("=" * 90)

📝 Launching Final Exam Grader Across All Folders
  Model Group Folder Name                    | Exam Samples | Used Threshold | True Final F1
------------------------------------------------------------------------------------------
  purine_nucleotide_metabolic_process_GO_... | 55           | 0.57           | 0.9437  
  antioxidant_activity_GO_0016209            | 147          | 0.89           | 0.8656  
  cation_channel_complex_GO_0034703          | 94           | 0.55           | 0.8632  
  molecular_carrier_activity_GO_0140104      | 127          | 0.90           | 0.8551  
  carbohydrate_derivative_binding_GO_0097367 | 122          | 0.84           | 0.8519  
  transferase_activity,_transferring_phos... | 24           | 0.66           | 0.8428  
  regulation_of_organelle_assembly_GO_190... | 49           | 0.63           | 0.8409  
  establishment_of_protein_localization_t... | 57           | 0.87           | 0.8332  
  RNA_nuclease_activity_GO_0004540           | 95           | 0

In [3]:
"""
================================================================================
NeuralProt — Performance Tier Distribution Chart
================================================================================
What this code does:
1. It reads your final exam scores from the computer's short-term memory.
2. It sets up 10 distinct "Score Drawers" (0-10%, 10-20% ... up to 90-100%).
3. It drops each model group into its correct performance drawer.
4. It prints a clean visual bar chart using blocks so you can instantly see 
   the mathematical spread of your models.
"""

# ── 1. INITIALIZE THE COUNTING DRAWERS ────────────────────────────────────────
# We create 10 empty lists to hold our groups
# Drawer 0 = 0-10%, Drawer 1 = 10-20%, ..., Drawer 9 = 90-100%
score_drawers = {i: [] for i in range(10)}

# ── 2. SORT THE SCORES INTO THE DRAWERS ───────────────────────────────────────
for item in final_test_report:
    # Turn the decimal score into a clean percentage number (e.g., 0.7563 -> 75.63)
    percentage_score = item["f1"] * 100
    
    # Use floor division to find the right drawer index number
    drawer_index = int(percentage_score // 10)
    
    # Safety Check: If a model scores a perfect 100%, put it in the 90-100% drawer safely
    if drawer_index == 10:
        drawer_index = 9
        
    # Put the item into that specific drawer pile
    score_drawers[drawer_index].append(item)

# ── 3. PRINT THE VISUAL CHART REPORT ──────────────────────────────────────────
print("=" * 75)
print("📊 NeuralProt — Final Exam Performance Distribution Chart")
print("=" * 75)
print(f"  {'Score Range':<12} | {'Total Models':<12} | Visual Spread Histogram")
print("-" * 75)

total_checked_sum = 0

for drawer_idx in sorted(score_drawers.keys()):
    low_boundary = drawer_idx * 10
    high_boundary = low_boundary + 10
    
    models_in_this_drawer = score_drawers[drawer_idx]
    count = len(models_in_this_drawer)
    total_checked_sum += count
    
    # Create a visual bar string using solid text blocks
    # Every block represents 1 model group sitting in this drawer
    visual_bar = "█" * count
    
    # Print the clean layout row line
    range_text = f"{low_boundary:>2}% to {high_boundary:>3}%"
    print(f"  {range_text:<12} | {count:>3} groups    | {visual_bar}")

print("=" * 75)
print(f"       Accounting Verification Sum: {total_checked_sum} total models categorized.")
print("=" * 75)


# ── 4. SNEAKY EXTRA: NAME LISTER FOR THE LOWEST TIERS ─────────────────────────
print("\n🚨 Quick Check: Which groups are sitting in the lowest 0% to 30% piles?")
found_low = False

for drawer_idx in [0, 1, 2]:
    for item in score_drawers[drawer_idx]:
        found_low = True
        print(f"   ⚠️ Low Tier ({drawer_idx*10}-{drawer_idx*10+10}%): {item['name']}")

if not found_low:
    print("   ✅ Amazing! Zero models scored below 30% on the final exam.")

📊 NeuralProt — Final Exam Performance Distribution Chart
  Score Range  | Total Models | Visual Spread Histogram
---------------------------------------------------------------------------
   0% to  10%  |   0 groups    | 
  10% to  20%  |   1 groups    | █
  20% to  30%  |   4 groups    | ████
  30% to  40%  |  22 groups    | ██████████████████████
  40% to  50%  |  42 groups    | ██████████████████████████████████████████
  50% to  60%  |  86 groups    | ██████████████████████████████████████████████████████████████████████████████████████
  60% to  70%  | 124 groups    | ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  70% to  80%  |  73 groups    | █████████████████████████████████████████████████████████████████████████
  80% to  90%  |  22 groups    | ██████████████████████
  90% to 100%  |   1 groups    | █
       Accounting Verification Sum: 375 total models categorized.

🚨 Quick Check: Which groups a

In [5]:
"""
================================================================================
NeuralProt — Upgraded Final Exam Evaluator (Fmax & Smin Engine)
================================================================================
What this code does:
1. It reads your master group map to dynamically discover all 370+ active groups.
2. It slices your data matrices to retrieve the locked-away 'test_idx.json' data piles.
3. It evaluates your model brains against a naive frequency baseline using 498 features.
4. It outputs your final macro-averaged Fmax and Smin scores for publication.
"""

import math
import csv


# ── 1. CONFIGURATION PATHS ────────────────────────────────────────────────────
PROCESSED_DIR   = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt_Beta/data/processed/processed_data"
MODEL_DIR = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt_Beta/backend/models"
OUTPUT_DIR      = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt_Beta/backend/test_evaluation"
ASSIGNMENT_PATH = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt_Beta/data/processed/go_group_assignment_v3.json"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Automatically detect every single active folder group in your project map
with open(ASSIGNMENT_PATH, "r") as f:
    _blueprint_data = json.load(f)
GROUPS = list(_blueprint_data["groups"].keys())

# ── 2. MODEL LAYER DEFINITION ─────────────────────────────────────────────────
class ProteinMLP(nn.Module):
    # FIXED: Doorway input dimension is now 498 to accept your upgraded features!
    def __init__(self, num_classes: int, input_dim: int = 498):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(256, num_classes),
        )
    def forward(self, x): return self.network(x)

# ── 3. REPRODUCIBLE SPLIT MATH ────────────────────────────────────────────────
def get_split_indices(n_total):
    indices = list(range(n_total))
    train_val_idx, test_idx = train_test_split(indices, test_size=0.1, random_state=42)
    train_idx, val_idx = train_test_split(train_val_idx, test_size=0.111, random_state=42)
    return train_idx, val_idx, test_idx

# ── 4. CAFA SCORE COMPUTATION ENGINE ─────────────────────────────────────────
class FmaxEngine:
    THRESHOLD_STEPS = 101
    def build_ic_table(self, label_matrix, go_terms):
        n = label_matrix.shape[0]
        if n == 0: return {}
        counts = label_matrix.sum(axis=0)
        return {t: -math.log2(counts[j]/n) if counts[j] > 0 else 0.0 for j, t in enumerate(go_terms)}

    def sweep(self, prob_matrix, y_true_matrix, go_terms, ic_table=None):
        thresholds = np.linspace(0.0, 1.0, self.THRESHOLD_STEPS)
        n_proteins, n_terms = prob_matrix.shape
        true_counts = y_true_matrix.sum(axis=1).astype(float)
        ic_array = np.array([ic_table.get(t, 0.0) if ic_table else 0.0 for t in go_terms])

        curve = {"threshold": [], "precision": [], "recall": [], "f": [], "s": []}
        for thresh in thresholds:
            pred = (prob_matrix >= thresh).astype(float)
            tp = (pred * y_true_matrix).sum(axis=1)
            n_pred = pred.sum(axis=1)
            
            avg_prec = (tp[n_pred > 0] / n_pred[n_pred > 0]).mean() if (n_pred > 0).sum() > 0 else 0.0
            avg_rec = np.where(true_counts > 0, tp / np.where(true_counts > 0, true_counts, 1.0), 0.0).mean()
            f = (2 * avg_prec * avg_rec / (avg_prec + avg_rec)) if (avg_prec + avg_rec > 0) else 0.0
            
            missed, wrong = y_true_matrix * (1 - pred), pred * (1 - y_true_matrix)
            ru = (missed * ic_array).sum(axis=1).mean()
            mi = (wrong * ic_array).sum(axis=1).mean()
            s = math.sqrt(ru**2 + mi**2)
            
            curve["threshold"].append(round(float(thresh), 4)); curve["f"].append(round(float(f), 6)); curve["s"].append(round(float(s), 6))
        
        fmax_idx, smin_idx = int(np.argmax(curve["f"])), int(np.argmin(curve["s"]))
        return {"fmax": curve["f"][fmax_idx], "smin": curve["s"][smin_idx]}

    def per_term_fmax(self, prob_col, y_true_col):
        if y_true_col.sum() == 0: return None
        best_f = 0.0
        for thresh in np.linspace(0.0, 1.0, self.THRESHOLD_STEPS):
            pred = (prob_col >= thresh).astype(int)
            tp, fp, fn = (pred * y_true_col).sum(), (pred * (1 - y_true_col)).sum(), ((1 - pred) * y_true_col).sum()
            p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            best_f = max(best_f, (2 * p * r / (p + r)) if (p + r) > 0 else 0.0)
        return round(best_f, 4)

# ── 5. RUN EXAM GRADER LOOP ───────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)
engine = FmaxEngine(); group_results = {}; all_term_rows = []

for group_name in GROUPS:
    # WINDOWS CHARACTER SAFETY FIXED
    safe_name = group_name.replace(":", "_").replace("/", "_")
    group_dir = os.path.join(PROCESSED_DIR, safe_name)
    model_path = os.path.join(MODEL_DIR, f"{safe_name}_best.pt")
    
    # FIXED: Looking for new compressed 'labels.npz' and 'terms.json' files
    if not os.path.exists(os.path.join(group_dir, "labels.npz")) or not os.path.exists(model_path):
        continue
        
    try:
        feature_matrix = np.load(os.path.join(group_dir, "features.npy"))
        with np.load(os.path.join(group_dir, "labels.npz")) as archive: label_matrix = archive["labels"]
        with open(os.path.join(group_dir, "terms.json")) as f: go_terms = json.load(f)
        
        train_idx, _, test_idx = get_split_indices(len(label_matrix))
        feature_test, label_test, label_train = feature_matrix[test_idx], label_matrix[test_idx], label_matrix[train_idx]
        
        # Inference prediction pass
        checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=False)
        model = ProteinMLP(num_classes=len(go_terms)).to(DEVICE)
        model.load_state_dict(checkpoint["model_state"] if isinstance(checkpoint, dict) and "model_state" in checkpoint else checkpoint)
        model.eval()
        
        with torch.no_grad():
            probs = torch.sigmoid(model(torch.tensor(feature_test, dtype=torch.float32).to(DEVICE))).cpu().numpy()
            
        ic_table = engine.build_ic_table(label_train, go_terms)
        dp_res = engine.sweep(probs, label_test, go_terms, ic_table)
        
        # Calculate Term Fmax scores for report card generation
        for j, go_term in enumerate(go_terms):
            support = int(label_test[:, j].sum())
            term_fmax = engine.per_term_fmax(probs[:, j], label_test[:, j])
            all_term_rows.append({"go_term": go_term, "group": group_name, "support": support, "fmax": term_fmax})
            
        group_results[group_name] = {"fmax": dp_res["fmax"], "smin": dp_res["smin"], "n_test": len(test_idx)}
    except Exception as e:
        print(f"❌ Muted folder calculation mismatch on: {safe_name} -> {e}")

# Save the necessary exam data outputs to disk
with open(os.path.join(OUTPUT_DIR, "test_fmax_summary.json"), "w") as f: json.dump(group_results, f, indent=2)
with open(os.path.join(OUTPUT_DIR, "test_per_term_fmax.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["go_term", "group", "support", "fmax"])
    w.writeheader(); w.writerows(all_term_rows)

print("✓ Final exam calculations complete! 'test_per_term_fmax.csv' safely generated.")

✓ Final exam calculations complete! 'test_per_term_fmax.csv' safely generated.


In [6]:
"""
================================================================================
NeuralProt — Upgraded Per-Term Report Card Breakdown
================================================================================
What this code does:
1. It opens the fresh exam output data file ('test_per_term_fmax.csv').
2. It fetches vocabulary terms from 'go_dict.json' to align true names.
3. It groups individual labels into Learnable, Weak, or Unlearnable piles.
"""

# ── 1. CONFIGURATION PATHS ────────────────────────────────────────────────────
TEST_EVAL_DIR = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt_Beta/backend/test_evaluation"
OUTPUT_DIR    = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt_Beta/backend/test_evaluation/per_term_analysis"
GO_DICT_PATH  = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt_Beta/data/processed/go_dict.json"

os.makedirs(OUTPUT_DIR, exist_ok=True)
csv_path = os.path.join(TEST_EVAL_DIR, "test_per_term_fmax.csv")

if not os.path.exists(csv_path):
    print("❌ ERROR: Cannot run analysis. 'test_per_term_fmax.csv' is missing from disk!")
else:
    with open(GO_DICT_PATH) as f: go_dict = json.load(f)
    
    rows = []
    with open(csv_path, newline="") as f:
        for r in csv.DictReader(f):
            rows.append({
                "go_term": r["go_term"], "group": r["group"],
                "support": int(r["support"]), "fmax": float(r["fmax"]) if r["fmax"] not in (None, "", "None") else 0.0
            })

    # Sort items by score barriers
    strong = [r for r in rows if r["support"] > 0 and r["fmax"] >= 0.70]
    moderate = [r for r in rows if r["support"] > 0 and 0.50 <= r["fmax"] < 0.70]
    weak = [r for r in rows if r["support"] > 0 and 0.01 <= r["fmax"] < 0.50]
    zero = [r for r in rows if r["support"] > 0 and r["fmax"] == 0.0]

    print("=" * 70)
    print("📊 NeuralProt — Per-Term Functional Learning Distribution")
    print("=" * 70)
    print(f"  -> Total Active Evaluated Labels: {len(rows):,}")
    print(f"  -> Strong Performers (Fmax >= 0.70) : {len(strong)} labels")
    print(f"  -> Moderate Performers (0.50-0.69)   : {len(moderate)} labels")
    print(f"  -> Weak/Lopsided Predictions         : {len(weak)} labels")
    print(f"  -> Blind/Unlearnable Predictions     : {len(zero)} labels")
    print("=" * 70)
    
    # Save the organized lists out as clean spreadsheets for your report documentation
    fields = ["go_term", "go_name", "group", "support", "fmax"]
    for label, pile in [("learnable_terms.csv", strong), ("weak_terms.csv", weak)]:
        with open(os.path.join(OUTPUT_DIR, label), "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=fields)
            w.writeheader()
            for r in pile:
                name = go_dict.get(r["go_term"], {}).get("name", "Unknown Job Function")
                w.writerow({"go_term": r["go_term"], "go_name": name, "group": r["group"], "support": r["support"], "fmax": r["fmax"]})
                
    print(f"✓ Spreadsheets successfully written to: {OUTPUT_DIR}/")

📊 NeuralProt — Per-Term Functional Learning Distribution
  -> Total Active Evaluated Labels: 10,821
  -> Strong Performers (Fmax >= 0.70) : 6584 labels
  -> Moderate Performers (0.50-0.69)   : 2022 labels
  -> Weak/Lopsided Predictions         : 1212 labels
  -> Blind/Unlearnable Predictions     : 0 labels
✓ Spreadsheets successfully written to: C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt_Beta/backend/test_evaluation/per_term_analysis/
